# 02 - Feature Engineering
## PhishScamSense: Real-Time Multimodal Phishing Defense

This notebook covers:
1. Loading the CIC-Bell-DNS2021 EDA sample from notebook 01
2. Lexical & structural feature extraction (23 features)
3. Shannon entropy and typosquatting indicators
4. Per-class feature distributions (4 classes: benign, phishing, malware, spam)
5. Feature correlation analysis across all classes

In [ ]:
import sys
import os
from urllib.parse import urlparse

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml.src.features.url_features import extract_url_features

sns.set_theme(style="whitegrid")

CLASS_NAMES  = ["benign", "phishing", "malware", "spam"]
CLASS_COLORS = ["#2ecc71", "#e74c3c", "#e67e22", "#9b59b6"]
CLASS_PALETTE = dict(zip(CLASS_NAMES, CLASS_COLORS))

PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
csv_path = os.path.join(PROCESSED_DIR, "eda_sample_with_features.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Loaded EDA sample: {df.shape}")
    print(df["label_name"].value_counts().reindex(CLASS_NAMES).to_string())
else:
    print("Run notebook 01 first to generate eda_sample_with_features.csv")

df.head()

## 2.1 Feature Extraction Functions

All 23 features used by the numerical branch (MLP/CapsNet) are implemented in
`ml/src/features/url_features.py` and imported above. This cell demonstrates each feature
on a real example from each class.

In [ ]:
# Show one representative URL per class and its extracted features
examples = (
    df.groupby("label_name", group_keys=False)
    .apply(lambda g: g.sample(1, random_state=0))
    .reset_index(drop=True)
)

for _, row in examples.iterrows():
    url = row["url"]
    feat = extract_url_features(url)
    print(f"\n[{row['label_name'].upper()}]  {url}")
    print(f"  {'Feature':<35} Value")
    print(f"  {'-'*50}")
    for k, v in feat.items():
        print(f"  {k:<35} {v}")

print(f"\nTotal features per URL: {len(feat)}")

## 2.2 Extract Features for Entire Dataset

## 2.2 Feature Matrix

The EDA sample loaded from notebook 01 already contains all 23 extracted features.
We select only the numerical feature columns here.

FEATURE_COLS = [
    "url_length", "hostname_length", "path_length", "num_dots", "num_hyphens",
    "num_underscores", "num_slashes", "num_query_params", "num_fragments",
    "num_digits", "num_special_chars", "url_entropy", "hostname_entropy",
    "has_ip_address", "has_punycode", "has_port", "has_https", "has_at_symbol",
    "has_double_slash_redirect", "subdomain_count", "tld_length",
    "consecutive_consonants_max", "vowel_ratio",
]

In [ ]:
FEATURE_COLS = [
    "url_length", "hostname_length", "path_length", "num_dots", "num_hyphens",
    "num_underscores", "num_slashes", "num_query_params", "num_fragments",
    "num_digits", "num_special_chars", "url_entropy", "hostname_entropy",
    "has_ip_address", "has_punycode", "has_port", "has_https", "has_at_symbol",
    "has_double_slash_redirect", "subdomain_count", "tld_length",
    "consecutive_consonants_max", "vowel_ratio",
]

features_df = df[FEATURE_COLS + ["label", "label_name"]].copy()
print(f"Feature matrix: {features_df.shape}")
features_df.head()

In [ ]:
## 2.3 Feature Correlation Matrix

fig, ax = plt.subplots(figsize=(14, 11))
corr = features_df[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    ax=ax, linewidths=0.4, annot_kws={"size": 7}, vmin=-1, vmax=1,
)
ax.set_title("Feature–Feature Correlation Matrix (all classes combined)", fontsize=13, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class mean feature values — ranks each feature by how much it differs across classes
per_class_means = features_df.groupby("label_name")[FEATURE_COLS].mean().T.reindex(columns=CLASS_NAMES)

# Feature discriminability: std of per-class means (high = more discriminative)
discriminability = per_class_means.std(axis=1).sort_values(ascending=False)

print("Features ranked by cross-class discriminability (std of per-class means):\n")
print(discriminability.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#e74c3c" if v > discriminability.median() else "#3498db"
          for v in discriminability.values]
discriminability.plot(kind="barh", ax=ax, color=colors[::-1])
ax.invert_yaxis()
ax.set_title("Feature Discriminability Across 4 Classes\n(std of per-class means — higher = more useful)")
ax.set_xlabel("Std of per-class mean")
ax.axvline(discriminability.median(), color="black", linewidth=0.8, linestyle="--", label="median")
ax.legend()
plt.tight_layout()
plt.show()

## 2.4 Feature Distributions by Class

In [ ]:
key_features = [
    "url_entropy", "hostname_entropy", "num_hyphens", "num_digits",
    "consecutive_consonants_max", "vowel_ratio",
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for name, color in CLASS_PALETTE.items():
        subset = features_df[features_df["label_name"] == name][feat]
        axes[i].hist(subset, bins=20, alpha=0.55, label=name, color=color)
    axes[i].set_title(feat, fontsize=11, fontweight="bold")
    axes[i].legend(fontsize=8)

plt.suptitle("Key Feature Distributions by Class (EDA sample)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Save feature matrix for use in training notebooks
out_path = os.path.join(PROCESSED_DIR, "features_dataset.csv")
features_df.to_csv(out_path, index=False)
print(f"Feature dataset saved → {out_path}")
print(f"Shape: {features_df.shape}")
print(f"Columns: {list(features_df.columns)}")